# Moon map downloader: preview/detail/ultra with target m/px

This notebook tests multiple download profiles and tries to pick the best available high-resolution WMS layer automatically.

In [32]:
import os
import random
import time
import math
import requests
import xml.etree.ElementTree as ET
from io import BytesIO
from PIL import Image

# --- Core config ---
SAVE_DIR = "lunar_surface_dataset_test"
TARGET_COUNT = 8

# If QUICK_COMPARE_MODES is not empty, these modes run one by one.
# If QUICK_COMPARE_MODES is empty, only DOWNLOAD_MODE runs.
DOWNLOAD_MODE = "ultra_087"
QUICK_COMPARE_MODES = ["preview", "detail", "ultra_087"]

# Optional manual layer override. Keep None to auto-pick a high-res layer.
FORCE_LAYER_NAME = None

# Use local AOI candidates first (recommended for high-res tests).
USE_AOI = True

# Filename coordinate style for quick copy/paste to QuickMap search box.
# Example tag: lat-lon_9.54888,24.28570  (lat,lon)
COORD_TAG_PREFIX = "lat-lon_"
COORD_DECIMALS = 5

# Stripe filter for bad WMS seams (vertical white bands).
STRIPE_FILTER_ENABLED = True
MAX_RETRIES_PER_TILE = 6
STRIPE_WHITE_THRESHOLD = 245
STRIPE_COLUMN_RATIO_THRESHOLD = 0.25
STRIPE_MIN_RUN_WIDTH = 8

# Prevent near-static random windows and duplicate coordinates per run.
MIN_BBOX_RANDOM_RANGE_DEG = 0.05
AVOID_DUPLICATE_COORDS = True

# Example AOIs (lon_min, lon_max, lat_min, lat_max) in EPSG:4326 degrees.
# First AOI is near the coordinate visible in your QuickMap screenshot.
AOI_CANDIDATES = [
    (-22.20, -21.70, 4.10, 4.60),
    (16.00, 16.80, -3.00, -2.20),
    (23.80, 24.60, 9.20, 10.00),
]

# Fallback global sampling bounds when AOI is disabled.
GLOBAL_BOUNDS = (-170.0, 160.0, -60.0, 60.0)

# WMS config
WMS_URL = "https://planetarymaps.usgs.gov/cgi-bin/mapserv"
WMS_MAP = "/maps/earth/moon_simp_cyl.map"
WMS_VERSION = "1.1.1"
WMS_SRS = "EPSG:4326"
REQUEST_TIMEOUT = 25

MODE_CONFIG = {
    "preview": {
        "image_size": 1024,
        "delta_deg": 2.0,
        "wms_format": "image/png",
        "file_ext": "png",
        "pil_format": "PNG",
        "sleep_seconds": 0.4,
    },
    "detail": {
        "image_size": 1536,
        "delta_deg": 0.80,
        "wms_format": "image/png",
        "file_ext": "png",
        "pil_format": "PNG",
        "sleep_seconds": 0.8,
    },
    "ultra_087": {
        "image_size": 2048,
        "target_mpp": 0.87,
        "wms_format": "image/png",
        "file_ext": "png",
        "pil_format": "PNG",
        "sleep_seconds": 1.2,
    },
}

if DOWNLOAD_MODE not in MODE_CONFIG:
    raise ValueError(f"Unknown DOWNLOAD_MODE: {DOWNLOAD_MODE}")

invalid = [m for m in QUICK_COMPARE_MODES if m not in MODE_CONFIG]
if invalid:
    raise ValueError(f"Unknown modes in QUICK_COMPARE_MODES: {invalid}")

os.makedirs(SAVE_DIR, exist_ok=True)
print("Config loaded.")

Config loaded.


In [33]:
MOON_RADIUS_M = 1737400.0
METERS_PER_DEG = (2.0 * 3.141592653589793 * MOON_RADIUS_M) / 360.0

def delta_from_target_mpp(target_mpp: float, image_size: int) -> float:
    return (target_mpp * image_size) / METERS_PER_DEG

def expected_mpp_equator(delta_deg: float, image_size: int) -> float:
    return (delta_deg * METERS_PER_DEG) / image_size

def local_name(tag: str) -> str:
    return tag.split('}')[-1]

def fetch_wms_capabilities(url: str, map_path: str) -> str:
    params = {
        "map": map_path,
        "service": "WMS",
        "version": WMS_VERSION,
        "request": "GetCapabilities",
    }
    r = requests.get(url, params=params, timeout=REQUEST_TIMEOUT)
    r.raise_for_status()
    return r.text

def parse_layers(cap_xml: str):
    root = ET.fromstring(cap_xml)
    layers = []
    for elem in root.iter():
        if local_name(elem.tag) != "Layer":
            continue

        name = None
        title = None
        for child in list(elem):
            lname = local_name(child.tag)
            if lname == "Name" and child.text:
                name = child.text.strip()
            elif lname == "Title" and child.text:
                title = child.text.strip()

        if name:
            layers.append({"name": name, "title": title or ""})

    return layers

def rank_layer(layer_name: str, layer_title: str) -> int:
    text = f"{layer_name} {layer_title}".lower()

    score = 0

    # Prefer NAC/ACT style high-res products.
    high_res_positive = ["nac", "act", "high", "meter", "0.5", "1m", "mosaic"]
    for kw in high_res_positive:
        if kw in text:
            score += 6

    # Penalize low-res global products.
    low_res_negative = ["wac", "low", "global", "shade"]
    for kw in low_res_negative:
        if kw in text:
            score -= 4

    # Bonus for explicit lunar references and common naming.
    if "lroc" in text:
        score += 2

    return score

def pick_layer(layers, forced_layer_name=None):
    names = {x["name"] for x in layers}
    if forced_layer_name:
        if forced_layer_name in names:
            return forced_layer_name
        raise ValueError(f"Forced layer not found: {forced_layer_name}")

    ranked = sorted(
        layers,
        key=lambda x: rank_layer(x["name"], x["title"]),
        reverse=True,
    )

    if not ranked:
        raise RuntimeError("No layers found in GetCapabilities response.")

    return ranked[0]["name"], ranked[:10]

def random_bbox_from_bounds(bounds, delta_deg):
    lon_min_b, lon_max_b, lat_min_b, lat_max_b = bounds

    if (lon_max_b - lon_min_b) <= delta_deg or (lat_max_b - lat_min_b) <= delta_deg:
        raise ValueError(
            f"Bounds too small for delta={delta_deg}: {bounds}. Expand bounds or reduce target m/px."
        )

    lon_min = random.uniform(lon_min_b, lon_max_b - delta_deg)
    lat_min = random.uniform(lat_min_b, lat_max_b - delta_deg)

    lon_max = lon_min + delta_deg
    lat_max = lat_min + delta_deg
    return lon_min, lat_min, lon_max, lat_max

cap_xml = fetch_wms_capabilities(WMS_URL, WMS_MAP)
all_layers = parse_layers(cap_xml)
selected_layer, ranked_preview = pick_layer(all_layers, FORCE_LAYER_NAME)

print(f"Selected layer: {selected_layer}")
print("Top layer candidates:")
for item in ranked_preview[:5]:
    print(f" - {item['name']} | {item['title']}")

Selected layer: KaguyaTC_Ortho
Top layer candidates:
 - KaguyaTC_Ortho | Kaguya TC Ortho Mosaic
 - Moon1M_Quads | Lunar 1M Quad Charts
 - uv_v2 | Clementine uv750 Global Basemap v2 Mosaic
 - uv_warp | Clementine uv750 Global Warped Mosaic
 - uv_lo | Clementine and Lunar Orbiter Hybrid Global Mosaic


In [34]:
active_modes = QUICK_COMPARE_MODES if QUICK_COMPARE_MODES else [DOWNLOAD_MODE]
print(f"Running modes: {', '.join(active_modes)}")

quickmap_rows = []

def detect_vertical_white_stripes(gray_img):
    width, height = gray_img.size
    px = gray_img.load()

    runs = []
    run_start = None

    for x in range(width):
        white_count = 0
        for y in range(height):
            if px[x, y] >= STRIPE_WHITE_THRESHOLD:
                white_count += 1

        col_ratio = white_count / height
        is_bad_col = col_ratio >= STRIPE_COLUMN_RATIO_THRESHOLD

        if is_bad_col and run_start is None:
            run_start = x
        elif not is_bad_col and run_start is not None:
            runs.append((run_start, x - 1))
            run_start = None

    if run_start is not None:
        runs.append((run_start, width - 1))

    wide_runs = [r for r in runs if (r[1] - r[0] + 1) >= STRIPE_MIN_RUN_WIDTH]

    edge_runs = []
    detect_thin_edges = globals().get("DETECT_THIN_EDGE_STRIPES", False)
    edge_max_width = globals().get("EDGE_STRIPE_MAX_WIDTH", 2)
    edge_border = globals().get("EDGE_STRIPE_BORDER_PIXELS", 2)

    if detect_thin_edges:
        for r in runs:
            run_w = r[1] - r[0] + 1
            touches_left = r[0] <= edge_border
            touches_right = r[1] >= (width - 1 - edge_border)
            if run_w <= edge_max_width and (touches_left or touches_right):
                edge_runs.append(r)

    defect_runs = wide_runs + edge_runs
    return (len(defect_runs) > 0), defect_runs

def build_quickmap_globe_url(lat_deg, lon_deg, mpp, pixel_size, fov_deg=60.0):
    # Build Cesium camera so link opens in 3D with approximate target m/px.
    moon_radius_m = MOON_RADIUS_M
    fov_rad = math.radians(fov_deg)
    tan_half_fov = math.tan(fov_rad / 2.0)
    if tan_half_fov <= 0:
        tan_half_fov = 0.57735026919  # tan(30 deg) fallback for fov=60
    altitude_m = (mpp * pixel_size) / (2.0 * tan_half_fov)

    lat = math.radians(lat_deg)
    lon = math.radians(lon_deg)

    nx = math.cos(lat) * math.cos(lon)
    ny = math.cos(lat) * math.sin(lon)
    nz = math.sin(lat)

    px = (moon_radius_m + altitude_m) * nx
    py = (moon_radius_m + altitude_m) * ny
    pz = (moon_radius_m + altitude_m) * nz

    dx, dy, dz = -nx, -ny, -nz

    ux = -math.sin(lat) * math.cos(lon)
    uy = -math.sin(lat) * math.sin(lon)
    uz = math.cos(lat)

    return (
        "https://quickmap.lroc.im-ldi.com/?camera="
        f"{px:.4f},{py:.4f},{pz:.4f},"
        f"{dx:.4f},{dy:.4f},{dz:.4f},"
        f"{ux:.4f},{uy:.4f},{uz:.4f},{fov_deg:.0f}"
        "&proj=22"
    )

for mode_name in active_modes:
    mode_cfg = MODE_CONFIG[mode_name]

    image_size = mode_cfg["image_size"]
    wms_image_format = mode_cfg["wms_format"]
    file_ext = mode_cfg["file_ext"]
    pil_save_format = mode_cfg["pil_format"]
    sleep_seconds = mode_cfg["sleep_seconds"]

    if "target_mpp" in mode_cfg:
        delta_deg = delta_from_target_mpp(mode_cfg["target_mpp"], image_size)
    else:
        delta_deg = mode_cfg["delta_deg"]

    mpp_eq = expected_mpp_equator(delta_deg, image_size)

    mode_save_dir = os.path.join(SAVE_DIR, mode_name) if len(active_modes) > 1 else SAVE_DIR
    os.makedirs(mode_save_dir, exist_ok=True)
    used_coord_keys = set()

    # Keep only AOIs that are large enough for current mode window size and allow true randomness.
    valid_aoi_bounds = [
        b for b in AOI_CANDIDATES
        if (b[1] - b[0]) > (delta_deg + MIN_BBOX_RANDOM_RANGE_DEG)
        and (b[3] - b[2]) > (delta_deg + MIN_BBOX_RANDOM_RANGE_DEG)
    ]

    global_ok = (
        (GLOBAL_BOUNDS[1] - GLOBAL_BOUNDS[0]) > (delta_deg + MIN_BBOX_RANDOM_RANGE_DEG)
        and (GLOBAL_BOUNDS[3] - GLOBAL_BOUNDS[2]) > (delta_deg + MIN_BBOX_RANDOM_RANGE_DEG)
    )

    if USE_AOI and not valid_aoi_bounds and not global_ok:
        raise ValueError(
            f"No AOI or global bounds can fit delta={delta_deg:.6f} with random margin. "
            "Increase bounds or lower target resolution."
        )

    if USE_AOI and not valid_aoi_bounds and global_ok:
        print(
            f"[warn] Mode {mode_name}: no AOI can fit delta={delta_deg:.6f}; "
            "falling back to GLOBAL_BOUNDS."
        )

    print(
        f"\n--- Mode={mode_name} | layer={selected_layer} | size={image_size} | delta={delta_deg:.6f} deg | requested~{mpp_eq:.3f} m/px(eq) ---"
    )

    for i in range(TARGET_COUNT):
        saved = False

        for attempt in range(1, MAX_RETRIES_PER_TILE + 1):
            if USE_AOI and valid_aoi_bounds:
                current_bounds = random.choice(valid_aoi_bounds)
            else:
                current_bounds = GLOBAL_BOUNDS

            lon_min, lat_min, lon_max, lat_max = random_bbox_from_bounds(current_bounds, delta_deg)

            lat_tag = f"{lat_min:.{COORD_DECIMALS}f}"
            lon_tag = f"{lon_min:.{COORD_DECIMALS}f}"
            coord_key = (lat_tag, lon_tag)

            if AVOID_DUPLICATE_COORDS and coord_key in used_coord_keys:
                print(
                    f"[{mode_name} {i+1}/{TARGET_COUNT}] retry {attempt}/{MAX_RETRIES_PER_TILE}: "
                    f"duplicate coords {lat_tag},{lon_tag}, redrawing..."
                )
                continue

            params = {
                "map": WMS_MAP,
                "request": "GetMap",
                "service": "WMS",
                "version": WMS_VERSION,
                "layers": selected_layer,
                "styles": "",
                "srs": WMS_SRS,
                "bbox": f"{lon_min},{lat_min},{lon_max},{lat_max}",
                "width": image_size,
                "height": image_size,
                "format": wms_image_format,
                "transparent": "false",
            }

            headers = {
                "User-Agent": "Mozilla/5.0"
            }

            try:
                response = requests.get(WMS_URL, params=params, headers=headers, timeout=REQUEST_TIMEOUT)
                response.raise_for_status()

                if "image" in response.headers.get("Content-Type", ""):
                    img = Image.open(BytesIO(response.content)).convert("L")

                    if STRIPE_FILTER_ENABLED:
                        has_stripes, stripe_runs = detect_vertical_white_stripes(img)
                        if has_stripes:
                            print(
                                f"[{mode_name} {i+1}/{TARGET_COUNT}] retry {attempt}/{MAX_RETRIES_PER_TILE}: "
                                f"detected vertical seams {stripe_runs}, redownloading..."
                            )
                            time.sleep(sleep_seconds)
                            continue

                    coord_tag = f"{COORD_TAG_PREFIX}{lat_tag},{lon_tag}"

                    lat_center = (lat_min + lat_max) / 2.0
                    lon_center = (lon_min + lon_max) / 2.0
                    mpp_local = mpp_eq * math.cos(math.radians(lat_center))

                    out_name = (
                        f"moon_{mode_name}_{i+1:03d}_mpp{mpp_local:.3f}_"
                        f"{coord_tag}.{file_ext}"
                    )
                    out_path = os.path.join(mode_save_dir, out_name)
                    img.save(out_path, pil_save_format)
                    used_coord_keys.add(coord_key)

                    quickmap_search = f"{lat_tag},{lon_tag}"
                    quickmap_url = build_quickmap_globe_url(
                        lat_center, lon_center, mpp_local, image_size
                    )
                    quickmap_rows.append(
                        {
                            "mode": mode_name,
                            "file_name": out_name,
                            "lat": lat_tag,
                            "lon": lon_tag,
                            "mpp": f"{mpp_local:.3f}",
                            "quickmap_search": quickmap_search,
                            "quickmap_url": quickmap_url,
                        }
                    )

                    print(
                        f"[{mode_name} {i+1}/{TARGET_COUNT}] Saved: {out_name} | "
                        f"QuickMap: {quickmap_search} | mpp={mpp_local:.3f}"
                    )

                    saved = True
                    break
                else:
                    snippet = response.text[:220].replace('\n', ' ')
                    print(f"[{mode_name} {i+1}/{TARGET_COUNT}] WMS non-image response: {snippet}")

            except Exception as e:
                print(f"[{mode_name} {i+1}/{TARGET_COUNT}] Request failed (attempt {attempt}): {e}")

            time.sleep(sleep_seconds)

        if not saved:
            print(f"[{mode_name} {i+1}/{TARGET_COUNT}] Failed after {MAX_RETRIES_PER_TILE} attempts.")

quickmap_log_path = os.path.join(SAVE_DIR, "quickmap_refs.tsv")
with open(quickmap_log_path, "w", encoding="utf-8") as f:
    f.write("mode\tfile_name\tlat\tlon\tmpp\tquickmap_search\tquickmap_url\n")
    for row in quickmap_rows:
        f.write(
            f"{row['mode']}\t{row['file_name']}\t{row['lat']}\t{row['lon']}\t"
            f"{row['mpp']}\t{row['quickmap_search']}\t{row['quickmap_url']}\n"
        )

print(f"\nDone. Files saved under: {SAVE_DIR}")
print(f"QuickMap log saved: {quickmap_log_path}")

Running modes: preview, detail, ultra_087
[warn] Mode preview: no AOI can fit delta=2.000000; falling back to GLOBAL_BOUNDS.

--- Mode=preview | layer=KaguyaTC_Ortho | size=1024 | delta=2.000000 deg | requested~59.225 m/px(eq) ---
[preview 1/8] Saved: moon_preview_001_mpp35.945_lat-lon_-53.63310,41.58658.png | QuickMap: -53.63310,41.58658 | mpp=35.945
[preview 2/8] retry 1/6: detected vertical seams [(0, 1023)], redownloading...
[preview 2/8] Saved: moon_preview_002_mpp47.257_lat-lon_36.06892,-116.73450.png | QuickMap: 36.06892,-116.73450 | mpp=47.257
[preview 3/8] Saved: moon_preview_003_mpp45.654_lat-lon_38.56969,-154.09967.png | QuickMap: 38.56969,-154.09967 | mpp=45.654
[preview 4/8] Saved: moon_preview_004_mpp59.002_lat-lon_-5.98051,91.46995.png | QuickMap: -5.98051,91.46995 | mpp=59.002
[preview 5/8] Saved: moon_preview_005_mpp55.586_lat-lon_19.19115,-27.90590.png | QuickMap: 19.19115,-27.90590 | mpp=55.586
[preview 6/8] retry 1/6: detected vertical seams [(449, 473), (953, 1012)